In [35]:
# --- SurrealDB Connection ---
DB_URL = "ws://35.209.122.206:8000" # Still use ws:// for the address
DB_NS = "partyscene"
DB_NAME = "partyscene"
DB_USER = "root"
DB_PASS = "rootrm"

# --- Table to Query ---
TABLE_NAME = "leads"

# --- The Key Encryption Key ---
# Use the hex string "6964fc08bae4d193"
KEK_HEX = "6964fc08bae4d193"
try:
    # Convert the hex string to bytes
    KEK_BYTES = KEK_HEX.encode()
    print(f"Using KEK Hex: {KEK_HEX}")
    print(f"Derived KEK Bytes (length {len(KEK_BYTES)}): {KEK_BYTES}")
    # The EnvelopeCipher class __init__ will handle padding this 8-byte key
except ValueError:
    print(f"ERROR: Invalid hexadecimal string provided for KEK: {KEK_HEX}")
    KEK_BYTES = None # Prevent execution if conversion fails

Using KEK Hex: 6964fc08bae4d193
Derived KEK Bytes (length 16): b'6964fc08bae4d193'


In [36]:

class EnvelopeCipher:
    """
    A secure two-way encryption class implementing envelope encryption.

    Envelope encryption is a security pattern in which a Data Encryption Key (DEK)
    is randomly generated for encrypting actual data, while the DEK itself is
    encrypted using a Key Encryption Key (KEK). The KEK should be securely stored
    in a secret manager (e.g., Google Cloud Secret Manager).

    Attributes:
        kek (bytes): The Key Encryption Key (KEK) used to encrypt and decrypt the DEK.

    Methods:
        encrypt(plaintext: bytes) -> dict:
            Encrypts the given plaintext using a random DEK and returns the encrypted data,
            the encrypted DEK, and their respective IVs.

        decrypt(encrypted_data: bytes, encrypted_dek: bytes, iv_data: bytes, iv_kek: bytes) -> bytes:
            Decrypts the encrypted DEK with the KEK and then uses it to decrypt the data.
    """

    def __init__(self, kek: bytes):
        assert isinstance(kek, bytes) and len(kek) in [
            16,
            24,
            32,
        ], "KEK must be a valid AES key"
        self.kek = kek
        self.backend = default_backend()

    def _get_cipher(self, key: bytes, iv: bytes):
        return ciphers.Cipher(
            ciphers.algorithms.AES(key), ciphers.modes.CBC(iv), backend=self.backend
        )

    def encrypt(self, plaintext: bytes):
        """
        Encrypts the plaintext using a randomly generated DEK, which is then encrypted using the KEK.

        Args:
            plaintext (bytes): The raw data to be encrypted.

        Returns:
            dict: Contains:
                - encrypted_data (bytes): The AES-encrypted data.
                - encrypted_dek (bytes): The AES-encrypted DEK using the KEK.
                - iv_data (bytes): IV used for encrypting the data.
                - iv_kek (bytes): IV used for encrypting the DEK.
        """
        decryption_key = os.urandom(32)
        data_initialization_vector = os.urandom(16)

        # Pad the data
        padder = padding.PKCS7(128).padder()
        padded_data = padder.update(plaintext) + padder.finalize()

        # Encrypt the data
        cipher_context = self._get_cipher(
            decryption_key, data_initialization_vector
        ).encryptor()
        encrypted_data = cipher_context.update(padded_data) + cipher_context.finalize()

        # Encrypt the DEK
        decryption_key_initialization_vector = os.urandom(16)
        cipher_context = self._get_cipher(
            self.kek, decryption_key_initialization_vector
        ).encryptor()

        padder = padding.PKCS7(128).padder()
        padded_dek = padder.update(decryption_key) + padder.finalize()
        encrypted_decryption_key = (
            cipher_context.update(padded_dek) + cipher_context.finalize()
        )

        return {
            "encrypted_data": encrypted_data,
            "encrypted_decryption_key": encrypted_decryption_key,
            "data_initialization_vector": data_initialization_vector,
            "decryption_key_initialization_vector": decryption_key_initialization_vector,
        }

    def decrypt(
        self,
        encrypted_data: bytes,
        encrypted_dek: bytes,
        data_initialization_vector: bytes,
        decryption_key_initialization_vector: bytes,
    ):
        """
        Decrypts the provided encrypted data using envelope decryption.

        Args:
            encrypted_data (bytes): The AES-encrypted data.
            encrypted_dek (bytes): The AES-encrypted DEK.
            iv_data (bytes): IV used during data encryption.
            iv_kek (bytes): IV used during DEK encryption.

        Returns:
            bytes: The original plaintext.
        """
        cipher_kek = self._get_cipher(
            self.kek, decryption_key_initialization_vector
        ).decryptor()
        padded_dek = cipher_kek.update(encrypted_dek) + cipher_kek.finalize()

        unpadder = padding.PKCS7(128).unpadder()
        dek = unpadder.update(padded_dek) + unpadder.finalize()

        cipher_data = self._get_cipher(dek, data_initialization_vector).decryptor()
        padded_data = cipher_data.update(encrypted_data) + cipher_data.finalize()

        unpadder = padding.PKCS7(128).unpadder()
        plaintext = unpadder.update(padded_data) + unpadder.finalize()
        return plaintext


In [37]:
from surrealdb import Surreal

import sys
import os

sys.path.append(os.path.abspath('/home/dylee/Documents/projects/partyscene/backstage/shared'))

from cryptography.hazmat.primitives import ciphers, padding
from cryptography.hazmat.backends import default_backend
from cryptography.exceptions import InvalidSignature

In [42]:
def run_decryption_test_sync():
    # Check if KEK conversion failed in Cell 2
    if KEK_BYTES is None:
        print("Cannot run test because KEK conversion failed.")
        return

    db = Surreal(DB_URL)
    fetched_record = None
    decrypted_text = None

    try:
        # 1. Connect and Sign In (Synchronous)
        print("Connecting to SurrealDB...")
        print("Signing in...")
        db.signin({"username": DB_USER, "password": DB_PASS}) # No await
        print("Using namespace/db...")
        db.use(DB_NS, DB_NAME) # No await
        print("Connected and authenticated.")

        # 2. Fetch one record (Synchronous)
        print(f"Fetching one record from '{TABLE_NAME}'...")
        query = f"SELECT * FROM {TABLE_NAME} LIMIT 1;"
        result = db.query(query) # No await

        # Extract the actual record from the response structure (same logic)
        print(f"Queried result: {result}")
        if len(result) > 0:
            fetched_record = result[0]
            print(f"Fetched record ID: {fetched_record.get('id', 'N/A')}")
        else:
            print(f"Could not find any records in '{TABLE_NAME}'.")
            return # Stop if no record found

        # 3. Attempt Decryption (if record found)
        if fetched_record:
            print("Attempting decryption...")
            try:
                # This will use the 8-byte key, which __init__ pads to 16 bytes
                # cipher = EnvelopeCipher(KEK_BYTES)

                # Extract fields, ensuring they are bytes (same logic)
                enc_data = fetched_record.get('encrypted_data')
                enc_dek = fetched_record.get('encrypted_decryption_key')
                data_iv = fetched_record.get('data_initialization_vector')
                dek_iv = fetched_record.get('decryption_key_initialization_vector')

                if not all([isinstance(f, bytes) for f in [enc_data, enc_dek, data_iv, dek_iv]]):
                     print("ERROR: One or more required encryption fields are missing or not bytes.")
                     # ... (error reporting) ...
                     return

                # Call decrypt (it's a synchronous method anyway)
                decrypted_text = EnvelopeCipher(KEK_BYTES).decrypt(
                        enc_data,
                        enc_dek,
                        data_iv,
                        dek_iv,
                    )
                print("--- DECRYPTION SUCCEEDED ---")
                try:
                    print("Decrypted Text (UTF-8 decoded):")
                    print(decrypted_text.decode('utf-8'))
                except UnicodeDecodeError:
                    print("Decrypted Bytes (could not decode as UTF-8):")
                    print(decrypted_text)
                print("--------------------------")

            except InvalidSignature:
                print("--- DECRYPTION FAILED: Invalid Padding ---")
                print(f"This suggests the KEK used ({KEK_HEX} padded to 16 bytes) is INCORRECT,")
                print("or the fetched encrypted data or IVs are corrupted.")
                print("-----------------------------------------")
            except Exception as e:
                print(f"--- DECRYPTION FAILED: Unexpected Error ---")
                print(f"Error: {e}")
                print("-----------------------------------------")

    except Exception as e:
        print(f"An error occurred during the DB interaction or decryption process: {e}")
    finally:
        # 4. Close connection (Synchronous)
        print("Closing connection...")
        db.close() # No await
        print("Connection closed.")

# --- Run the synchronous test function ---
run_decryption_test_sync()

Connecting to SurrealDB...
Signing in...
Using namespace/db...
Connected and authenticated.
Fetching one record from 'leads'...
Queried result: [{'data_initialization_vector': b'\x92\xc5_\xebi\x96\xfd)\x1c\xe6\x16p\xda\xf1\xb2\x95', 'decryption_key_initialization_vector': b'i\x16e\xe7N\xc0\xf0\xda\xeb\xd1.\xab\xa7\xb3C\x14', 'encrypted_data': b'\x83"\xbc\x98\x8e\xa8\x7f\xf7\xf6\xdcy\xfc3\xf4.t', 'encrypted_decryption_key': b',P\xf9/V\xa9\x94\xb3\x13\xd3\x08\xa3\xd6\x9e\x86\xc0\xb1?-i\xbb\xc7{V\xf6I\x1f\x9a\xa2\x8eO;\xe3\x83\xa7\xbf\xfa\xb3\xa8\x06\xff\xf7\xfc\xd9\xb2\x08\xba\xc6', 'id': RecordID(table_name=leads, record_id=3oc3rn0n8jh3vf3nyqex), 'usecase': 'early access tester'}]
Fetched record ID: leads:3oc3rn0n8jh3vf3nyqex
Attempting decryption...
--- DECRYPTION SUCCEEDED ---
Decrypted Text (UTF-8 decoded):
test@gmail.com
--------------------------
Closing connection...
Connection closed.
